# Tai video tu vatlythaynang.com

Khoa hoc #4: **Nen tang toan cho Vat ly (MIEN PHI)**
Ma: `Toanchovatly26_27` | 269 hoc sinh | 11 video (~27h)

Video hosted on Bunny Stream (HLS/m3u8). Documents hosted on R2.

In [ ]:
import requests
import json
import re
import os
import subprocess
from pathlib import Path
from playwright.sync_api import sync_playwright

## 1. Config

In [ ]:
EMAIL = "admin@vatlythaynang.com"PASSWORD = "admin123456"COURSE_ID = 4BASE_URL = "https://vatlythaynang.com"OUTPUT_DIR = Path("../data/videos")

## 2. Login & Scrape Course Structure

Trang dung Next.js App Router voi React Server Actions. Data nam trong RSC payload.

In [ ]:
def scrape_course_structure(course_id: int) -> list:    """Login via Playwright, fetch course page, extract tree nodes from RSC payload."""    with sync_playwright() as p:        browser = p.chromium.launch(headless=True)        page = browser.new_context().new_page()        page.goto(f"{BASE_URL}/auth/login", wait_until="networkidle")        page.fill('input[type="email"]', EMAIL)        page.fill('input[type="password"]', PASSWORD)        page.click('button[type="submit"]')        page.wait_for_url("**/admin/**", timeout=15000)        print(f"[OK] Logged in -> {page.url}")        page.goto(f"{BASE_URL}/admin/courses/{course_id}?tab=structure", wait_until="networkidle")        page.wait_for_timeout(3000)        html = page.content()        browser.close()    scripts = re.findall(r'self\.__next_f\.push\(\[1,"(.*?)"\]\)', html, re.DOTALL)    nodes = []    for script in scripts:        decoded = script.encode().decode("unicode_escape")        if '"nodes"' not in decoded:            continue        idx = decoded.find('"nodes":[')        if idx < 0:            continue        bc = 0        start = idx + 8        for j in range(start, len(decoded)):            if decoded[j] == "[":                bc += 1            elif decoded[j] == "]":                bc -= 1                if bc == 0:                    nodes = json.loads(decoded[start:j+1])                    break        break    print(f"[OK] Found {len(nodes)} top-level folders")    return nodes

In [ ]:
nodes = scrape_course_structure(COURSE_ID)for node in nodes:    title = node.get("title", "?")    children = node.get("children", [])    vids = [c for c in children if c.get("fileKind") == "VIDEO"]    docs = [c for c in children if c.get("fileKind") == "DOCUMENT"]    print(f"\n  {title}")    for v in vids:        dur = v.get("durationSeconds", 0) or 0        print(f"    VIDEO: {v['title']} ({dur//3600}h{(dur%3600)//60:02d}m)")    for d in docs:        print(f"    DOC:   {d.get('title', '?')}")

## 3. Flatten files

In [ ]:
def flatten_files(nodes: list) -> list:    """Flatten tree into flat list with folder context."""    files = []    for node in nodes:        folder = node.get("title", "unknown")        for child in node.get("children", []):            files.append({                "folder": folder,                "id": child["id"],                "title": child.get("title", "?"),                "kind": child.get("fileKind"),                "duration": child.get("durationSeconds") or 0,                "video_url": child.get("videoUrl"),                "file_url": child.get("fileUrl"),                "bunny_video_id": child.get("bunnyVideoId"),            })    return filesall_files = flatten_files(nodes)video_files = [f for f in all_files if f["kind"] == "VIDEO"]doc_files = [f for f in all_files if f["kind"] == "DOCUMENT"]print(f"Total: {len(all_files)} files ({len(video_files)} videos, {len(doc_files)} docs)")total_dur = sum(f["duration"] for f in video_files)print(f"Total video duration: {total_dur//3600}h{(total_dur%3600)//60:02d}m")

## 4. Download functions

In [ ]:
def sanitize(name: str) -> str:    """Remove special characters from filename."""    return re.sub(r"[^\w\s\-.]", "", name).strip().replace(" ", "_")def download_video(url: str, output: Path, timeout: int = 900) -> bool:    """Download HLS video via ffmpeg. Falls back to re-encode if copy fails."""    cmd = ["ffmpeg", "-y", "-i", url, "-c", "copy", "-bsf:a", "aac_adtstoasc", str(output)]    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)    if r.returncode == 0:        return True    cmd2 = ["ffmpeg", "-y", "-i", url, "-c:v", "libx264", "-preset", "fast", "-c:a", "aac", str(output)]    r2 = subprocess.run(cmd2, capture_output=True, text=True, timeout=timeout)    return r2.returncode == 0def download_doc(url: str, output: Path) -> bool:    """Download PDF document via requests."""    resp = requests.get(url, timeout=60)    if resp.status_code == 200:        output.write_bytes(resp.content)        return True    return False

## 5. Download videos

In [ ]:
for f in video_files:    folder = OUTPUT_DIR / sanitize(f["folder"])    folder.mkdir(parents=True, exist_ok=True)    output = folder / (sanitize(f["title"]) + ".mp4")    if output.exists() and output.stat().st_size > 1_000_000:        print(f"SKIP: {output.name} ({output.stat().st_size // 1_048_576}MB)")        continue    dur = f["duration"]    print(f"Downloading: {f['title']} ({dur//3600}h{(dur%3600)//60:02d}m) ...", flush=True)    ok = download_video(f["video_url"], output)    if ok:        print(f"  OK: {output.stat().st_size // 1_048_576}MB")    else:        print(f"  FAILED: {f['title']}")

## 6. Download documents

In [ ]:
for f in doc_files:    folder = OUTPUT_DIR / sanitize(f["folder"])    folder.mkdir(parents=True, exist_ok=True)    output = folder / (sanitize(f["title"]) + ".pdf")    if output.exists() and output.stat().st_size > 1_000:        print(f"SKIP: {output.name}")        continue    print(f"Downloading: {f['title']} ...", flush=True)    ok = download_doc(f["file_url"], output)    if ok:        print(f"  OK: {output.stat().st_size // 1_024}KB")    else:        print(f"  FAILED")

## 7. Summary

In [ ]:
print("=" * 60)print("DOWNLOAD SUMMARY")print("=" * 60)total_size = 0for folder in sorted(OUTPUT_DIR.iterdir()):    if not folder.is_dir():        continue    files = list(folder.glob("*"))    folder_size = sum(f.stat().st_size for f in files)    total_size += folder_size    print(f"\n  {folder.name}/ ({len(files)} files, {folder_size // 1_048_576}MB)")    for f in sorted(files):        print(f"    {f.name} ({f.stat().st_size // 1_024}KB)")print(f"\n  TOTAL: {total_size // 1_048_576}MB")